# LeetCode 929 — Unique Email Addresses bằng Python

**Link đề:** https://leetcode.com/problems/unique-email-addresses/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table (set), String

```python
class Solution:
    def numUniqueEmails(self, emails: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ quy tắc chuẩn hóa email: `.` bị bỏ qua và `+...` bị cắt — **chỉ trong local name**.
2. Thành thạo các thao tác chuỗi: `split`, `partition`, `find`, `replace`, slicing.
3. Dùng `set` để **đếm số phần tử khác nhau** (khử trùng lặp).
4. Đi từ cách "so từng cặp" → chuẩn hóa + `set` → duyệt ký tự thủ công (không dùng hàm có sẵn).
5. Trace, chứng minh, Time/Space, các lỗi kinh điển (áp dụng quy tắc lên cả domain).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — có bao nhiêu hộp thư thật sự nhận thư?

Mỗi email gồm **local name** và **domain name**, ngăn cách bởi `@`. Ví dụ `alice@leetcode.com`: local = `alice`, domain = `leetcode.com`.

Quy tắc chuyển tiếp (chỉ áp dụng cho **local name**):
1. Dấu `.` trong local name bị **bỏ qua**: `alice.z@leetcode.com` → `alicez@leetcode.com`.
2. Mọi thứ **sau dấu `+` đầu tiên** trong local name bị **bỏ**: `m.y+name@email.com` → `my@email.com`.
3. Hai quy tắc dùng cùng lúc được. Domain **giữ nguyên**.

Cho `emails`, mỗi địa chỉ nhận đúng 1 thư. Trả về **số địa chỉ khác nhau thực sự nhận thư**.

| `emails` | Output |
|---|:---:|
| `["test.email+alex@leetcode.com","test.e.mail+bob.cathy@leetcode.com","testemail+david@lee.tcode.com"]` | `2` |
| `["a@leetcode.com","b@leetcode.com","c@leetcode.com"]` | `3` |

Ví dụ 1: hai email đầu → `testemail@leetcode.com`; email thứ ba → `testemail@lee.tcode.com` (domain khác vì `.` trong domain **không** bị bỏ).

**Constraints:** `1 <= emails.length <= 100`, `1 <= emails[i].length <= 100`; chỉ chữ thường, `+`, `.`, `@`; đúng một `@`; local và domain không rỗng; local không bắt đầu bằng `+`; domain kết thúc `.com`.

## 2. Nền tảng Python cần dùng

### 2.1. `str.split(sep)` — tách chuỗi

`"a@b.com".split("@")` → `["a", "b.com"]`. Nếu có nhiều dấu phân cách, ta được nhiều phần. Có thể **mở gói** ngay: `local, domain = email.split("@")` (chỉ an toàn khi chắc chắn có đúng 1 `@` — đề đảm bảo).

In [1]:
email = "test.e.mail+bob.cathy@leetcode.com"
local, domain = email.split("@")
print("local =", local)
print("domain =", domain)
print("a+b+c".split("+"), "<- tách mọi dấu +")

local = test.e.mail+bob.cathy
domain = leetcode.com
['a', 'b', 'c'] <- tách mọi dấu +


### 2.2. Cắt phần sau `+` đầu tiên

Ba cách:
- `local.split("+")[0]` — lấy phần trước `+` đầu tiên (nếu không có `+`, trả nguyên chuỗi).
- `local.partition("+")[0]` — `partition` luôn trả **3 phần** `(trước, sep, sau)` và chỉ tách ở lần xuất hiện **đầu tiên**.
- `local.find("+")` trả vị trí `+` đầu tiên, hoặc `-1` nếu không có → kết hợp slicing.

In [2]:
for local in ["test.e.mail+bob.cathy", "abc", "a+b+c"]:
    p = local.find("+")
    print(f"{local!r:24} split->{local.split('+')[0]!r:14} partition->{local.partition('+')!r:32} find->{p}")

'test.e.mail+bob.cathy'  split->'test.e.mail'  partition->('test.e.mail', '+', 'bob.cathy') find->11
'abc'                    split->'abc'          partition->('abc', '', '')                  find->-1
'a+b+c'                  split->'a'            partition->('a', '+', 'b+c')                find->1


### 2.3. `str.replace(old, new)` — bỏ dấu chấm

`"t.e.st".replace(".", "")` → `"test"`. `replace` trả **chuỗi mới** (chuỗi bất biến), không sửa chuỗi gốc.

In [3]:
s = "t.e.st"
t = s.replace(".", "")
print(s, "->", t)

t.e.st -> test


### 2.4. `set` — tập hợp các phần tử khác nhau

`set` giống `dict` chỉ có key: mỗi phần tử xuất hiện **tối đa một lần**. `add` để thêm, `len` để đếm, `in` để kiểm tra (trung bình `O(1)`).

In [4]:
seen = set()
for addr in ["x@a.com", "y@a.com", "x@a.com"]:
    seen.add(addr)
    print("thêm", addr, "->", seen)
print("số phần tử khác nhau:", len(seen))
print("'x@a.com' in seen?", "x@a.com" in seen)

thêm x@a.com -> {'x@a.com'}
thêm y@a.com -> {'y@a.com', 'x@a.com'}
thêm x@a.com -> {'y@a.com', 'x@a.com'}
số phần tử khác nhau: 2
'x@a.com' in seen? True


### 2.5. Set comprehension

`{f(x) for x in data}` tạo set trực tiếp — giống list comprehension nhưng dùng ngoặc nhọn.

In [5]:
words = ["Ab", "ab", "AB", "c"]
print({w.lower() for w in words})
print(len({w.lower() for w in words}))

{'c', 'ab'}
2


## 3. Cách trực giác — chuẩn hóa thủ công rồi so từng cặp

Người mới có thể nghĩ: chuẩn hóa từng email, rồi với mỗi email kiểm tra "đã có email chuẩn hóa nào giống nó trước đó chưa" bằng cách **duyệt lại danh sách**.

Trước hết viết hàm chuẩn hóa `normalize(email)`.

In [6]:
def normalize(email: str) -> str:
    local, domain = email.split("@")
    local = local.split("+")[0]      # quy tắc 2: cắt sau '+'
    local = local.replace(".", "")   # quy tắc 1: bỏ '.'
    return local + "@" + domain

for e in ["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]:
    print(f"{e:38} -> {normalize(e)}")

test.email+alex@leetcode.com           -> testemail@leetcode.com
test.e.mail+bob.cathy@leetcode.com     -> testemail@leetcode.com
testemail+david@lee.tcode.com          -> testemail@lee.tcode.com


In [7]:
def num_unique_pairwise(emails: list[str]) -> int:
    normalized = [normalize(e) for e in emails]
    count = 0
    for i in range(len(normalized)):
        is_new = True
        for j in range(i):                 # so với mọi email trước nó
            if normalized[j] == normalized[i]:
                is_new = False
                break
        if is_new:
            count += 1
    return count

print(num_unique_pairwise(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]))
print(num_unique_pairwise(["a@leetcode.com", "b@leetcode.com", "c@leetcode.com"]))

2
3


### 3.1. Thứ tự áp dụng quy tắc có quan trọng không?

Cắt `+` trước hay bỏ `.` trước đều cho cùng kết quả, vì bỏ `.` không tạo/xóa dấu `+`, và cắt `+` chỉ bỏ phần đuôi. Kiểm chứng:

In [8]:
local = "m.y+na.me"
a = local.split("+")[0].replace(".", "")
b_ = local.replace(".", "").split("+")[0]
print(a, b_, a == b_)

my my True


## 4. Nút thắt và insight

Cách so từng cặp tốn `O(N²)` phép so sánh chuỗi (`N` = số email). Với `N = 100` vẫn ổn, nhưng ý tưởng không mở rộng được.

**Insight:** "đếm số giá trị **khác nhau**" ⇒ dùng **`set`**. Mỗi `add` và kiểm tra trùng chỉ tốn trung bình `O(L)` (băm chuỗi độ dài `L`), không cần so với từng phần tử trước.

**Mô hình chung:** `số phần tử khác nhau = len({chuẩn_hóa(x) for x in data})`.

Bài toán thực chất chỉ là **viết đúng hàm chuẩn hóa** — phần còn lại là `set`.

## 5. Lời giải chính — chuẩn hóa + set

**Mảnh 1:** `unique = set()`.

**Mảnh 2:** với mỗi email: tách `local, domain`.

**Mảnh 3:** `local = local.split("+")[0].replace(".", "")`.

**Mảnh 4:** `unique.add(local + "@" + domain)`.

**Mảnh 5:** `return len(unique)`.

In [9]:
def num_unique_set(emails: list[str]) -> int:
    unique = set()
    for email in emails:
        local, domain = email.split("@")
        local = local.split("+")[0].replace(".", "")
        unique.add(local + "@" + domain)
    return len(unique)

print(num_unique_set(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]))
print(num_unique_set(["a@leetcode.com", "b@leetcode.com", "c@leetcode.com"]))

2
3


`local.split("+")[0].replace(".", "")` là **gọi chuỗi phương thức** (method chaining): kết quả của `split(...)[0]` là một chuỗi, ta gọi tiếp `.replace(...)` trên chuỗi đó.

## 6. TRACE từng bước

Ví dụ 1 của đề:

| # | email | local gốc | sau cắt `+` | sau bỏ `.` | domain | địa chỉ chuẩn | set sau khi add |
|:---:|---|---|---|---|---|---|---|
| 1 | `test.email+alex@leetcode.com` | `test.email+alex` | `test.email` | `testemail` | `leetcode.com` | `testemail@leetcode.com` | {A} |
| 2 | `test.e.mail+bob.cathy@leetcode.com` | `test.e.mail+bob.cathy` | `test.e.mail` | `testemail` | `leetcode.com` | `testemail@leetcode.com` | {A} (trùng) |
| 3 | `testemail+david@lee.tcode.com` | `testemail+david` | `testemail` | `testemail` | `lee.tcode.com` | `testemail@lee.tcode.com` | {A, B} |

Kết quả `len = 2`.

In [10]:
def trace_unique_emails(emails):
    unique = set()
    for k, email in enumerate(emails, start=1):
        local, domain = email.split("@")
        cut = local.split("+")[0]
        nodot = cut.replace(".", "")
        addr = nodot + "@" + domain
        status = "TRÙNG" if addr in unique else "MỚI"
        unique.add(addr)
        print(f"#{k}: local={local!r} -> cắt+ {cut!r} -> bỏ. {nodot!r} | domain={domain!r}")
        print(f"     địa chỉ chuẩn = {addr!r} [{status}] | số địa chỉ = {len(unique)}")
    return len(unique)

trace_unique_emails(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"])

#1: local='test.email+alex' -> cắt+ 'test.email' -> bỏ. 'testemail' | domain='leetcode.com'
     địa chỉ chuẩn = 'testemail@leetcode.com' [MỚI] | số địa chỉ = 1
#2: local='test.e.mail+bob.cathy' -> cắt+ 'test.e.mail' -> bỏ. 'testemail' | domain='leetcode.com'
     địa chỉ chuẩn = 'testemail@leetcode.com' [TRÙNG] | số địa chỉ = 1
#3: local='testemail+david' -> cắt+ 'testemail' -> bỏ. 'testemail' | domain='lee.tcode.com'
     địa chỉ chuẩn = 'testemail@lee.tcode.com' [MỚI] | số địa chỉ = 2


2

In [11]:
# Ví dụ có '+' và '.' "nguy hiểm" nằm trong domain
trace_unique_emails(["a.b+c@x.y.com", "ab@x.y.com", "ab@xy.com", "ab@x+y.com"])

#1: local='a.b+c' -> cắt+ 'a.b' -> bỏ. 'ab' | domain='x.y.com'
     địa chỉ chuẩn = 'ab@x.y.com' [MỚI] | số địa chỉ = 1
#2: local='ab' -> cắt+ 'ab' -> bỏ. 'ab' | domain='x.y.com'
     địa chỉ chuẩn = 'ab@x.y.com' [TRÙNG] | số địa chỉ = 1
#3: local='ab' -> cắt+ 'ab' -> bỏ. 'ab' | domain='xy.com'
     địa chỉ chuẩn = 'ab@xy.com' [MỚI] | số địa chỉ = 2
#4: local='ab' -> cắt+ 'ab' -> bỏ. 'ab' | domain='x+y.com'
     địa chỉ chuẩn = 'ab@x+y.com' [MỚI] | số địa chỉ = 3


3

## 7. Các lời giải khác và bảng so sánh

### 7.1. Dùng `partition` và `find` (không tạo list trung gian)

In [12]:
def num_unique_partition(emails: list[str]) -> int:
    unique = set()
    for email in emails:
        local, _, domain = email.partition("@")
        plus = local.find("+")
        if plus != -1:
            local = local[:plus]
        unique.add(local.replace(".", "") + "@" + domain)
    return len(unique)

print(num_unique_partition(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]))

2


`_` là tên biến quy ước cho giá trị **không dùng tới** (ở đây là chính dấu `@`).

### 7.2. Duyệt từng ký tự thủ công (không dùng hàm chuỗi)

Hữu ích khi phỏng vấn yêu cầu "tự xử lý": duyệt từng ký tự với cờ trạng thái:
- Gặp `@` → từ đây là domain, chép nguyên phần còn lại.
- Gặp `+` khi đang ở local → bỏ qua tới `@`.
- Gặp `.` khi đang ở local → bỏ qua.

In [13]:
def normalize_manual(email: str) -> str:
    out = []
    i = 0
    n = len(email)
    while i < n and email[i] != "@":
        ch = email[i]
        if ch == "+":
            while email[i] != "@":       # nhảy tới '@'
                i += 1
            break
        if ch != ".":
            out.append(ch)
        i += 1
    out.append(email[i:])                # '@' + domain giữ nguyên
    return "".join(out)

def num_unique_manual(emails: list[str]) -> int:
    return len({normalize_manual(e) for e in emails})

for e in ["test.e.mail+bob.cathy@leetcode.com", "a.b@c.d.com", "x+@y.com"]:
    print(f"{e:36} -> {normalize_manual(e)}")

test.e.mail+bob.cathy@leetcode.com   -> testemail@leetcode.com
a.b@c.d.com                          -> ab@c.d.com
x+@y.com                             -> x@y.com


### 7.3. Regex: xóa `.` và cắt `+...` trong local

`re.sub(pattern, repl, s)` thay mọi đoạn khớp `pattern` bằng `repl`. Ta áp dụng chỉ trên **local**:
- `re.sub(r"\+.*", "", local)` — xóa từ `+` đầu tiên đến hết (`.*` = "mọi ký tự, bao nhiêu cũng được").
- `re.sub(r"\.", "", ...)` — xóa mọi dấu chấm (`\.` vì `.` trong regex nghĩa là "ký tự bất kỳ").

In [14]:
import re

def num_unique_regex(emails: list[str]) -> int:
    unique = set()
    for email in emails:
        local, domain = email.split("@")
        local = re.sub(r"\.", "", re.sub(r"\+.*", "", local))
        unique.add(local + "@" + domain)
    return len(unique)

print(re.sub(r"\+.*", "", "a.b+c+d"), "|", re.sub(r"\.", "", "a.b.c"), "|", re.sub(".", "", "a.b.c"), "<- quên \\ thì xóa sạch!")
print(num_unique_regex(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]))

a.b | abc |  <- quên \ thì xóa sạch!
2


### 7.4. Bảng so sánh

`N` = số email, `L` = độ dài tối đa một email.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Chuẩn hóa + so từng cặp | `O(N·L + N²·L)` | `O(N·L)` | baseline |
| **split/replace + set** | `O(N·L)` trung bình | `O(N·L)` | nên nộp |
| partition/find + set | `O(N·L)` | `O(N·L)` | ít list trung gian |
| Duyệt ký tự thủ công | `O(N·L)` | `O(N·L)` | tự kiểm soát, hợp phỏng vấn |
| Regex | `O(N·L)` | `O(N·L)` | gọn, cần biết escape |

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def numUniqueEmails(self, emails: list[str]) -> int:
        unique = set()
        for email in emails:
            local, domain = email.split("@")
            local = local.split("+")[0].replace(".", "")
            unique.add(local + "@" + domain)
        return len(unique)

sol = Solution()
assert sol.numUniqueEmails(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]) == 2
assert sol.numUniqueEmails(["a@leetcode.com", "b@leetcode.com", "c@leetcode.com"]) == 3
print("Solution PASS 2 ví dụ đề bài")

Solution PASS 2 ví dụ đề bài


## 9. Vì sao thuật toán đúng?

**Hàm chuẩn hóa đúng:** theo đề, địa chỉ thực nhận thư của `local@domain` là `f(local)@domain`, trong đó `f` bỏ phần từ `+` đầu tiên và bỏ mọi `.`. Hàm `normalize` làm đúng `f` trên local và giữ nguyên domain.

**Hai email gửi tới cùng hộp thư ⇔ cùng địa chỉ chuẩn hóa.** Vì địa chỉ chuẩn hóa không còn `.` hay `+` trong local, nó là **dạng chính tắc** (canonical form): mỗi hộp thư có đúng một dạng.

**Đếm đúng:** `set` chứa mỗi địa chỉ chuẩn hóa đúng một lần (invariant: sau khi xử lý `k` email đầu, `unique` = tập địa chỉ chuẩn hóa của `k` email đó). Vậy `len(unique)` = số hộp thư khác nhau. ∎

**Lưu ý domain:** domain không bị chuẩn hóa, nên `lee.tcode.com` ≠ `leetcode.com` — khớp với ví dụ 1.

## 10. Phân tích Time / Space

Gọi `N = len(emails)`, `L` = độ dài tối đa của một email (`≤ 100`).

- **Time:** mỗi email: `split`, `replace`, nối chuỗi đều `O(L)`; `set.add` cần băm chuỗi `O(L)` (trung bình). Tổng `O(N·L)`.
- **Space:** set chứa tối đa `N` chuỗi, mỗi chuỗi `≤ L` → `O(N·L)`.

So với baseline: phần so từng cặp tốn thêm `O(N²·L)`.

In [16]:
import time, random
rng = random.Random(929)
def rand_email():
    local = "".join(rng.choice("ab.+") for _ in range(rng.randint(1, 30))).lstrip("+") or "a"
    domain = "".join(rng.choice("ab.") for _ in range(rng.randint(1, 5))) + ".com"
    return local + "@" + domain
big = [rand_email() for _ in range(3000)]
for f in [num_unique_pairwise, num_unique_set]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{f.__name__:20} -> {r}, {time.perf_counter() - t0:.4f}s")

num_unique_pairwise  -> 1789, 0.0884s
num_unique_set       -> 1789, 0.0017s


`.lstrip("+")` bỏ các dấu `+` ở **đầu** chuỗi (đề nói local không bắt đầu bằng `+`); `or "a"` thay chuỗi rỗng bằng `"a"`.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Hai ví dụ đề.
2. Một email.
3. `.` và `+` trong **domain** — không được chuẩn hóa.
4. Nhiều dấu `+`; `+` ngay sau ký tự đầu; `.` liên tiếp.
5. Local giống nhau, domain khác nhau.

In [17]:
tests = [
    (["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"], 2),
    (["a@leetcode.com", "b@leetcode.com", "c@leetcode.com"], 3),
    (["a@x.com"], 1),
    (["a.b@x.com", "ab@x.com", "a..b@x.com"], 1),
    (["ab@x.y.com", "ab@xy.com"], 2),
    (["ab@x+y.com", "ab@x.com"], 2),
    (["a+b+c@x.com", "a@x.com", "a+@x.com"], 1),
    (["a.+.b@x.com", "a@x.com"], 1),
    (["test.email+alex@leetcode.com", "test.email.leet+alex@code.com"], 2),
    (["ab@x.com", "ab@y.com", "a.b@y.com"], 2),
]
solvers = [num_unique_pairwise, num_unique_set, num_unique_partition, num_unique_manual,
           num_unique_regex, Solution().numUniqueEmails]
for emails, expected in tests:
    for f in solvers:
        got = f(emails)
        assert got == expected, (f.__name__, emails, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 10 test x 6 cách


In [18]:
# Ngẫu nhiên có seed: mọi cách cho cùng kết quả
for _ in range(300):
    emails = [rand_email() for _ in range(rng.randint(1, 20))]
    expected = num_unique_pairwise(emails)
    for f in solvers[1:]:
        assert f(emails) == expected, (f.__name__, emails)
print("PASS 300 bộ ngẫu nhiên")

PASS 300 bộ ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Bỏ `.` trên **toàn bộ** email (cả domain)

In [19]:
def wrong_replace_all(emails):
    unique = set()
    for e in emails:
        e = e.replace(".", "")
        local, domain = e.split("@")
        unique.add(local.split("+")[0] + "@" + domain)
    return len(unique)

ex1 = ["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]
print("wrong_replace_all(ex1) =", wrong_replace_all(ex1), "| đúng: 2")

wrong_replace_all(ex1) = 1 | đúng: 2


**Vì sao sai:** `lee.tcode.com` và `leetcode.com` bị gộp thành một. **Sửa:** tách `@` **trước**, chỉ sửa local.

### Lỗi 2: Cắt `+` trên toàn bộ email

In [20]:
def wrong_plus_all(emails):
    unique = set()
    for e in emails:
        e = e.split("+")[0]            # mất luôn '@domain'!
        unique.add(e.replace(".", ""))
    return len(unique)

print("wrong_plus_all(ex1) =", wrong_plus_all(ex1), "| đúng: 2")

wrong_plus_all(ex1) = 1 | đúng: 2


**Vì sao sai:** `split("+")[0]` trên cả email làm mất phần `@domain` → mọi email có cùng phần đầu bị coi là giống nhau dù domain khác. **Sửa:** chỉ cắt trong local.

### Lỗi 3: Dùng `split("+")[1]` hoặc lấy phần sau `+`

Đề nói **bỏ** phần sau `+`, giữ phần **trước** → `[0]`.

### Lỗi 4: Dùng list thay set, quên khử trùng

In [21]:
def wrong_list(emails):
    out = []
    for e in emails:
        local, domain = e.split("@")
        out.append(local.split("+")[0].replace(".", "") + "@" + domain)
    return len(out)

print("wrong_list(ex1) =", wrong_list(ex1), "| đúng: 2")

wrong_list(ex1) = 3 | đúng: 2


### Lỗi 5: `str.replace` mà không gán lại

`local.replace(".", "")` **không sửa** `local` (chuỗi bất biến). Phải viết `local = local.replace(".", "")`.

In [22]:
local = "a.b"
local.replace(".", "")
print("không gán lại:", local)
local = local.replace(".", "")
print("có gán lại   :", local)

không gán lại: a.b
có gán lại   : ab


## 13. Mẹo nhận dạng pattern

- "Đếm số giá trị **khác nhau** sau khi áp quy tắc biến đổi" → **chuẩn hóa về dạng chính tắc** + `set`.
- "Quy tắc chỉ áp dụng cho **một phần** chuỗi" → **tách phần trước**, xử lý riêng, rồi ghép lại.
- "Bỏ phần sau ký tự X đầu tiên" → `split(X)[0]`, `partition(X)[0]` hoặc `find` + slicing.

Bài cùng khuôn "chuẩn hóa + set/dict": 49 Group Anagrams (chuẩn hóa = sắp xếp ký tự), 804 Unique Morse Code Words, 1078/893 Groups of Special-Equivalent Strings, 609 Find Duplicate File in System.

## 14. Biến thể

### 14.1. Trả về **danh sách nhóm** email cùng hộp thư

Dùng `dict` ánh xạ địa chỉ chuẩn → danh sách email gốc. `defaultdict(list)` tự tạo list rỗng khi key mới.

In [23]:
from collections import defaultdict

def group_emails(emails):
    groups = defaultdict(list)
    for e in emails:
        groups[normalize(e)].append(e)
    return dict(groups)

for k, v in group_emails(ex1).items():
    print(k, "<-", v)

testemail@leetcode.com <- ['test.email+alex@leetcode.com', 'test.e.mail+bob.cathy@leetcode.com']
testemail@lee.tcode.com <- ['testemail+david@lee.tcode.com']


### 14.2. Hộp thư nhận **nhiều thư nhất**

In [24]:
from collections import Counter
cnt = Counter(normalize(e) for e in ex1)
print(cnt.most_common(1))

[('testemail@leetcode.com', 2)]


### 14.3. Domain không phân biệt hoa/thường

Nếu đề cho phép chữ hoa và quy định domain không phân biệt hoa/thường, chuẩn hóa thêm `domain.lower()`.

In [25]:
def normalize_ci(email):
    local, domain = email.split("@")
    return local.split("+")[0].replace(".", "") + "@" + domain.lower()

print(len({normalize_ci(e) for e in ["ab@X.com", "a.b@x.COM"]}))   # 1

1


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
"a+b+c@d.com".split("@")[0].split("+")[0]
"x.y.z".partition(".")
len({"a", "b", "a"})
```

### Bài 2 — Viết hàm chuẩn hóa

In [26]:
def exercise_normalize(email: str) -> str:
    # TODO: tách @, cắt +, bỏ ., ghép lại
    pass

# assert exercise_normalize("test.e.mail+bob.cathy@leetcode.com") == "testemail@leetcode.com"
# assert exercise_normalize("a@b.c.com") == "a@b.c.com"

### Bài 3 — Tự viết lời giải 929

In [27]:
def exercise_num_unique(emails: list[str]) -> int:
    # TODO: set + chuẩn hóa
    pass

# assert exercise_num_unique(["test.email+alex@leetcode.com", "test.e.mail+bob.cathy@leetcode.com", "testemail+david@lee.tcode.com"]) == 2

### Bài 4 — Chuẩn hóa không dùng `split/replace/partition/find`

In [28]:
def exercise_normalize_manual(email: str) -> str:
    # TODO: duyệt từng ký tự với cờ trạng thái
    pass

# assert exercise_normalize_manual("m.y+name@email.com") == "my@email.com"

### Bài 5 — Unique Morse Code Words (LeetCode 804)

Mỗi chữ `a..z` có mã Morse (cho sẵn). Đếm số **phiên dịch** Morse khác nhau của danh sách từ.

In [29]:
MORSE = [".-","-...","-.-.","-..",".","..-.","--.","....","..",".---","-.-",".-..","--",
         "-.","---",".--.","--.-",".-.","...","-","..-","...-",".--","-..-","-.--","--.."]

def exercise_unique_morse(words: list[str]) -> int:
    # TODO: chuẩn hóa mỗi từ thành chuỗi Morse, đếm bằng set
    pass

# assert exercise_unique_morse(["gin", "zen", "gig", "msg"]) == 2
# assert exercise_unique_morse(["a"]) == 1

### Bài 6 — Mini challenge: Group Anagrams (LeetCode 49)

Gom các từ là hoán vị chữ cái của nhau. Trả về số nhóm và danh sách nhóm (thứ tự tùy ý).

In [30]:
def exercise_group_anagrams(words: list[str]) -> list[list[str]]:
    # TODO: chuẩn hóa = "".join(sorted(w)), dùng defaultdict(list)
    pass

# groups = exercise_group_anagrams(["eat","tea","tan","ate","nat","bat"])
# assert sorted(sorted(g) for g in groups) == [["ate","eat","tea"], ["bat"], ["nat","tan"]]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `split("@")[0]` = `"a+b+c"` | partition tách lần đầu | set khử trùng |
| 2 | 3 bước theo đề | `local, domain = email.split("@")` | `local.split("+")[0].replace(".", "")` |
| 3 | dạng chính tắc + set | `unique.add(...)` | `return len(unique)` |
| 4 | duyệt tới `@` | gặp `+` thì nhảy tới `@` | xem mục 7.2 |
| 5 | chuẩn hóa = dịch Morse | `MORSE[ord(c) - ord('a')]` | `len({"".join(...) for w in words})` |
| 6 | key = chữ cái đã sắp xếp | `defaultdict(list)` | `list(groups.values())` |

## 17. Đáp án tham khảo

Bài 1: `'a'`, `('x', '.', 'y.z')`, `2`.

In [31]:
print("a+b+c@d.com".split("@")[0].split("+")[0], "x.y.z".partition("."), len({"a", "b", "a"}))

a ('x', '.', 'y.z') 2


In [32]:
def answer_normalize(email):
    local, domain = email.split("@")
    return local.split("+")[0].replace(".", "") + "@" + domain

def answer_num_unique(emails):
    return len({answer_normalize(e) for e in emails})

def answer_normalize_manual(email):
    out = []
    in_local, skipping = True, False
    for ch in email:
        if not in_local:
            out.append(ch)
        elif ch == "@":
            in_local = False
            out.append(ch)
        elif skipping or ch == ".":
            continue
        elif ch == "+":
            skipping = True
        else:
            out.append(ch)
    return "".join(out)

def answer_unique_morse(words):
    return len({"".join(MORSE[ord(c) - ord("a")] for c in w) for w in words})

def answer_group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

assert answer_normalize("test.e.mail+bob.cathy@leetcode.com") == "testemail@leetcode.com"
assert answer_normalize("a@b.c.com") == "a@b.c.com"
assert answer_num_unique(ex1) == 2
assert answer_normalize_manual("m.y+name@email.com") == "my@email.com"
assert answer_normalize_manual("a+b@x+y.com") == "a@x+y.com"
assert answer_unique_morse(["gin", "zen", "gig", "msg"]) == 2
assert answer_unique_morse(["a"]) == 1
groups = answer_group_anagrams(["eat","tea","tan","ate","nat","bat"])
assert sorted(sorted(g) for g in groups) == [["ate","eat","tea"], ["bat"], ["nat","tan"]]
for emails, expected in tests:
    assert answer_num_unique(emails) == expected
    assert len({answer_normalize_manual(e) for e in emails}) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numUniqueEmails(self, emails: list[str]) -> int:
        unique = set()
        for email in emails:
            local, domain = email.split("@")
            local = local.split("+")[0].replace(".", "")
            unique.add(local + "@" + domain)
        return len(unique)
```

**Câu thần chú:** *"Tách @ trước, sửa local sau (cắt +, bỏ .), domain để yên, đếm bằng set."*

**Checklist trước khi submit:**
- [ ] Tách `@` **trước** khi áp dụng quy tắc.
- [ ] Lấy phần **trước** `+` (`[0]`), không phải sau.
- [ ] Không đụng vào domain (kể cả `.` hay `+` trong domain).
- [ ] Gán lại kết quả `replace` (chuỗi bất biến).
- [ ] Dùng `set` để khử trùng, trả `len`.